# CoDA-Synth — Traditional ML (Colab)

**Stylometric detection of LLM-rewritten dark-web documents with sparse features.**

Research use only. Public corpora (CoDA / DUTA) or this proxy. **Do not scrape live hidden services.** Do not reproduce operational criminal instructions. Synthetic class = paraphrase of public text.

This notebook is the method in the project guide: word TF–IDF, character n-grams, 11 stylometric features, LinearSVC + logistic regression, id-safe split, ablations for RQ1–RQ4.

In [ ]:
# CPU is enough for TF–IDF + LinearSVC. GPU is optional and only for a local rewrite model.
%pip -q install scikit-learn==1.5.2 pandas numpy scipy matplotlib seaborn joblib

## Ethics lock

- Use corpora released for research. Cite Jin et al. (2022) / Al-Nabki et al. when you use their text.
- Do not unmask vendors, wallets, or victims. Keep CoDA masks intact.
- Synthesis prompt must paraphrase existing public text only — no new criminal how-to.
- If a document looks like CSAM or trafficking, drop the row and do not inspect further.

In [ ]:
import re, json, os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, f1_score, accuracy_score, ConfusionMatrixDisplay, RocCurveDisplay
from scipy.sparse import hstack, csr_matrix
import matplotlib.pyplot as plt

ROOT = os.environ.get("CODA_ROOT", "/content/coda_synth")
for d in ["data", "models", "outputs"]:
    os.makedirs(f"{ROOT}/{d}", exist_ok=True)

TOPICS = ["Arms","Crypto","Drugs","Electronics","Financial","Gambling","Hacking","Pornography","Violence","Others"]
STOP = set(ENGLISH_STOP_WORDS)
TAG = re.compile(r"<[^>]+>")
WS = re.compile(r"\s+")

def clean_text(s):
    s = TAG.sub(" ", str(s))
    return WS.sub(" ", s).strip()

## Proxy corpus (runs without CoDA access)

CoDA is gated (`s2w-ai/CoDA`). Until Hugging Face accepts a research request, this cell builds a **synthetic proxy** with the same ten topic names, masked identifiers, and a controlled rewrite protocol. Swap in real CoDA later — the rest of the notebook does not change.

In [ ]:
PROMPT = '''Rewrite the following document in a different wording and sentence rhythm.
Keep the same facts, names that are already masked, and the same topic.
Do not add products, prices, instructions, or new claims.
Do not mention that you are an AI.

DOCUMENT:
'''

# Tiny ethical proxy: marketplace-style logistics + topic nouns. Not a dump of hidden-service pages.
BANKS = {
    "Electronics": ["wts laptop 15 inch i7 16gb. screen pretty clean. pm [PERSON]. escrow only.",
                    "selling used phone unlocked 128gb. battery around 86 percent. local near [ADDR]."],
    "Crypto": ["need to move a small bag to [WALLET] tonight. fees on my side. ping [CONTACT].",
               "desk rates for the usual pairs. invoice already written. reference [ID]."],
    "Gambling": ["mirror of the sportsbook is up. same book, new hostname. ping [CONTACT] if it 404s.",
                 "cashout queue is slow. payout to the same method only."],
    "Financial": ["desk is open for posted pairs. same-day if you ping before 14:00. use [ID].",
                  "invoice question: fourteen day window. paperwork ready. write [CONTACT]."],
    "Drugs": ["same listing family as last week. stock note only. title has the weight. pm listing code.",
              "pack log: tuesday went out. check tracking before you write in caps. [ADDR]."],
    "Arms": ["catalog note for replica and display pieces. no live advice. serial photos masked as [ID].",
             "surplus list: cases, slings, cleaning kits. paperwork stays with the item."],
    "Hacking": ["notes from a tabletop writeup. logs redacted. no exploit code. quote finding [ID].",
                "second pair of eyes on a scope document. target is an internal lab box."],
    "Pornography": ["adult catalog metadata only. aliases already masked as [PERSON]. no media in this post.",
                    "clip index for a takedown match. hashes only. duration and tags on the row."],
    "Violence": ["shock-clip index for a moderation queue. labels only, no graphic descriptions. hash [ID].",
                 "takedown list from a mirror that rehosted news footage. hashes and titles."],
    "Others": ["misc stall: hoodie, cables, a small hosting coupon. local pickup near [ADDR].",
               "lost and found for a meetup badge. name on it is [PERSON]."],
}

def polish(text):
    t = re.sub(r"\s+", " ", text).strip()
    t = t.replace("wts", "Offering").replace("pm", "please message").replace("dont", "do not")
    if not t.endswith("."): t += "."
    return t[0].upper() + t[1:]

def rhythm(text):
    bits = [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]
    out = []
    for s in bits:
        w = s.split()
        if len(w) > 8:
            mid = len(w)//2
            out += [" ".join(w[:mid]), " ".join(w[mid:])]
        else:
            out.append(s)
    t = ". ".join(b[0].upper()+b[1:] if b else b for b in out) + "."
    return t.replace(" pm ", " message ")

rng = np.random.default_rng(42)
rows = []
for topic in TOPICS:
    for i in range(24):
        base = rng.choice(BANKS[topic])
        extra = " escrow only. padded mailer. if you flake twice i move on. write [CONTACT]."
        text = clean_text(base + extra)
        doc_id = f"coda-{topic[:4].lower()}-{i:03d}"
        rows.append(dict(doc_id=doc_id, text=text, topic=topic, source="human", generator="original"))
        if rng.random() < 0.5:
            gen = "rhythm-B" if rng.random() < 0.45 else "polish-A"
            new = rhythm(text) if gen == "rhythm-B" else polish(text)
            rows.append(dict(doc_id=doc_id, text=new, topic=topic, source="llm", generator=gen))

syn = pd.DataFrame(rows)
syn["n_tok"] = syn.text.str.split().str.len()
syn = syn[syn.n_tok >= 12].copy()
syn.to_csv(f"{ROOT}/data/coda_synth.csv", index=False)
print(syn.groupby(["source","topic"]).size().unstack(fill_value=0))
print("rows", len(syn), "ids", syn.doc_id.nunique())

## Optional: load real CoDA after HF access

Uncomment after Hugging Face accepts the request. Map columns to `doc_id, text, topic`. Do not mix DUTA labels without a mapping table.

In [ ]:
# from huggingface_hub import login
# from google.colab import userdata
# login(token=userdata.get("HF_TOKEN"))
# from datasets import load_dataset
# ds = load_dataset("s2w-ai/CoDA")
# print(ds)
# print(ds["train"].column_names)  # map to doc_id, text, topic

## Id-safe split

A document and its rewrite share content. Split on **original doc_id** first (70 / 15 / 15), stratify by topic, then attach each id’s rows to the same split.

In [ ]:
meta = syn.drop_duplicates("doc_id")[["doc_id","topic"]]
id_train, id_tmp = train_test_split(meta["doc_id"], test_size=0.30, random_state=42, stratify=meta["topic"])
id_val, id_test = train_test_split(id_tmp, test_size=0.50, random_state=42,
    stratify=meta.set_index("doc_id").loc[id_tmp, "topic"])

def tag(frame, ids, name):
    out = frame[frame.doc_id.isin(ids)].copy()
    out["split"] = name
    return out

full = pd.concat([
    tag(syn, id_train, "train"),
    tag(syn, id_val, "val"),
    tag(syn, id_test, "test"),
], ignore_index=True)
full.to_parquet(f"{ROOT}/data/splits.parquet")
print(full.split.value_counts().to_dict())

## Features + LinearSVC

In [ ]:
def stylo(text):
    t = text or ""
    toks = re.findall(r"\b\w+\b", t.lower())
    n = max(len(toks), 1)
    sents = [s for s in re.split(r"[.!?]+", t) if s.strip()]
    ns = max(len(sents), 1)
    uniq = len(set(toks))
    hap = sum(1 for w in set(toks) if toks.count(w) == 1)
    return np.array([
        n,
        np.mean([len(w) for w in toks]) if toks else 0,
        np.mean([len(s.split()) for s in sents]) if sents else 0,
        uniq / n,
        hap / n,
        sum(w in STOP for w in toks) / n,
        sum(ch in ".,;:!?-()" for ch in t) / max(len(t), 1),
        sum(ch.isdigit() for ch in t) / max(len(t), 1),
        sum(ch.isupper() for ch in t) / max(len(t), 1),
        t.count("!") / ns,
        t.count("?") / ns,
    ], dtype=float)

tr = full[full.split=="train"]; va = full[full.split=="val"]; te = full[full.split=="test"]

word = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.9, max_features=20000, sublinear_tf=True)
char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=2, max_features=12000)
Xw_tr, Xw_va, Xw_te = word.fit_transform(tr.text), word.transform(va.text), word.transform(te.text)
Xc_tr, Xc_va, Xc_te = char.fit_transform(tr.text), char.transform(va.text), char.transform(te.text)

S_tr = np.vstack(tr.text.map(stylo)); S_va = np.vstack(va.text.map(stylo)); S_te = np.vstack(te.text.map(stylo))
scaler = StandardScaler()
S_tr, S_va, S_te = scaler.fit_transform(S_tr), scaler.transform(S_va), scaler.transform(S_te)

def pack(w, c, s):
    return hstack([w, c, csr_matrix(s)]).tocsr()

X_tr, X_va, X_te = pack(Xw_tr,Xc_tr,S_tr), pack(Xw_va,Xc_va,S_va), pack(Xw_te,Xc_te,S_te)
y_src_tr = (tr.source=="llm").astype(int)
y_src_va = (va.source=="llm").astype(int)
y_src_te = (te.source=="llm").astype(int)
le = LabelEncoder()
y_top_tr = le.fit_transform(tr.topic)
y_top_va = le.transform(va.topic)
y_top_te = le.transform(te.topic)

src = LinearSVC(class_weight="balanced", random_state=42)
src.fit(X_tr, y_src_tr)
print("SOURCE val\n", classification_report(y_src_va, src.predict(X_va), digits=3, target_names=["human","llm"]))

logit = LogisticRegression(class_weight="balanced", solver="saga", max_iter=2000, random_state=42)
logit.fit(X_tr, y_src_tr)

top = LinearSVC(class_weight="balanced", random_state=42)
top.fit(X_tr, y_top_tr)
print("TOPIC val\n", classification_report(y_top_va, top.predict(X_va), digits=3, target_names=le.classes_))

## Evaluation, plots, ablations

In [ ]:
pred_s = src.predict(X_te)
pred_t = top.predict(X_te)
print(classification_report(y_src_te, pred_s, target_names=["human","llm"], digits=3))
print("source AUC", roc_auc_score(y_src_te, logit.decision_function(X_te)))
print("topic weighted F1", f1_score(y_top_te, pred_t, average="weighted"))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(y_src_te, pred_s, display_labels=["human","llm"], ax=ax[0], colorbar=False)
ConfusionMatrixDisplay.from_predictions(y_top_te, pred_t, display_labels=le.classes_, ax=ax[1], colorbar=False, xticks_rotation=45)
ax[0].set_title("Source"); ax[1].set_title("Topic")
fig.tight_layout()
fig.savefig(f"{ROOT}/outputs/confusion.png", dpi=160)

def eval_stack(use_char, use_stylo):
    parts_tr, parts_te = [Xw_tr], [Xw_te]
    if use_char:
        parts_tr.append(Xc_tr); parts_te.append(Xc_te)
    if use_stylo:
        parts_tr.append(csr_matrix(S_tr)); parts_te.append(csr_matrix(S_te))
    Xt, Xe = hstack(parts_tr).tocsr(), hstack(parts_te).tocsr()
    m = LinearSVC(class_weight="balanced", random_state=42).fit(Xt, y_src_tr)
    lg = LogisticRegression(class_weight="balanced", solver="saga", max_iter=1500, random_state=42).fit(Xt, y_src_tr)
    pred = m.predict(Xe)
    return dict(
        source_f1=f1_score(y_src_te, pred),
        source_auc=roc_auc_score(y_src_te, lg.decision_function(Xe)),
        topic_wf1=f1_score(y_top_te, LinearSVC(class_weight="balanced", random_state=42).fit(Xt, y_top_tr).predict(Xe), average="weighted"),
    )

ablation = pd.DataFrame([
    {"stack":"word", **eval_stack(False, False)},
    {"stack":"word+char", **eval_stack(True, False)},
    {"stack":"word+char+stylo", **eval_stack(True, True)},
])
ablation.to_csv(f"{ROOT}/outputs/ablation.csv", index=False)
print(ablation)

## Citations

- Jin, Y., Jang, E., Lee, Y., Shin, S., & Chung, J.-W. (2022). Shedding New Light on the Language of the Dark Web. NAACL.
- Al-Nabki, M. W., Fidalgo, E., Alegre, E., & Fernández-Robles, L. (2017/2019). DUTA / DUTA-10K.
- Jin et al. (2023). DarkBERT. ACL. Related work — you are not required to run DarkBERT.

Scope is frozen: public data, traditional ML, two tasks, Colab. If CoDA access lags, start on this proxy and swap files later.